In [ ]:
# INSTALL LIBRARIES
!pip install -q pymupdf python-pptx
print("libraries installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.6 MB/s eta 0:00:00
libraries installed


In [ ]:
## CONVERTOR - SELECT TEXT OR IMAGE MODE

import io
import re
import fitz  # PyMuPDF
from pptx import Presentation
from pptx.util import Inches, Pt

def _set_slide_size(prs, w_in, h_in):
    prs.slide_width = Inches(w_in)
    prs.slide_height = Inches(h_in)

def _page_inches(page):
    r = page.rect
    return r.width / 72.0, r.height / 72.0

def _render_page_png(page, dpi=200, transparent=False):
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=transparent)
    return pix.tobytes("png")

def _guess_title_and_body_from_page(page):
    """
    Use PyMuPDF's structured 'dict' text to guess a title (largest font span)
    and collect the rest as body text.
    """
    d = page.get_text("dict")
    spans = []
    for b in d.get("blocks", []):
        for l in b.get("lines", []):
            for s in l.get("spans", []):
                text = s.get("text", "").strip()
                if text:
                    spans.append({
                        "text": text,
                        "size": s.get("size", 12),
                        "font": s.get("font", ""),
                        "bbox": s.get("bbox", None),
                    })

    if not spans:
        return None, ""

    # Find largest font size span as title candidate
    max_size = max(s["size"] for s in spans)
    # Be a bit tolerant to avoid picking random giant glyphs
    title_candidates = [s for s in spans if s["size"] >= max_size - 0.5 and len(s["text"]) <= 120]
    title = title_candidates[0]["text"] if title_candidates else None

    # Build body from all spans, keeping reading order
    # Join spans back into lines/paragraph-ish chunks
    # (simplified: merge by sentence-like boundaries)
    body_text = " ".join(s["text"] for s in spans if s["text"] != title).strip()

    # Light cleanup of excessive spaces
    body_text = re.sub(r"[ \t]+", " ", body_text)
    # Break at sentence-ish boundaries to avoid a mega paragraph
    body_text = re.sub(r"(?<=[.!?])\s+(?=[A-Z0-9])", "\n", body_text)

    return title, body_text

def convert_pdf_to_pptx(
    pdf_path: str,
    pptx_path: str = "converted.pptx",
    mode: str = "image",            # "image" or "text"
    dpi: int = 220,
    match_size: bool = False,
    margin_in: float = None,
    transparent: bool = False,
    layout: str = "widescreen"       # for image mode when not matching size: "widescreen" or "standard"
):
    """
    Convert a PDF to PPTX, one slide per page.

    mode="image": renders each page as an image (best fidelity).
    mode="text":  extracts text into editable text boxes (layout approximate).
    """
    if margin_in is None:
        margin_in = 0.0 if match_size else 0.5  # a little breathing room for default slides

    prs = Presentation()
    blank = prs.slide_layouts[6]  # empty layout

    # Set a default slide size (only used when not matching size per-page)
    if not match_size:
        if layout == "standard":
            _set_slide_size(prs, 10.0, 7.5)     # 4:3
        else:
            _set_slide_size(prs, 13.333, 7.5)   # 16:9

    doc = fitz.open(pdf_path)
    if doc.page_count == 0:
        raise ValueError("PDF has no pages.")

    for i in range(doc.page_count):
        page = doc[i]

        # If matching size, adjust slide to the page size
        if match_size:
            w_in, h_in = _page_inches(page)
            # Keep inside PowerPoint practical limits
            w_in = min(max(w_in, 1.0), 56.0)
            h_in = min(max(h_in, 1.0), 56.0)
            _set_slide_size(prs, w_in, h_in)

        slide = prs.slides.add_slide(blank)

        if mode == "image":
            img_bytes = _render_page_png(page, dpi=dpi, transparent=transparent)
            stream = io.BytesIO(img_bytes)

            # Fit the image within the slide minus margins while keeping aspect
            slide_w = prs.slide_width
            slide_h = prs.slide_height
            left = Inches(margin_in)
            top  = Inches(margin_in)
            avail_w = slide_w - Inches(margin_in*2)
            avail_h = slide_h - Inches(margin_in*2)

            # Add first at full width, PowerPoint will preserve aspect and let us clip by specifying only width or height.
            pic = slide.shapes.add_picture(stream, left, top, width=avail_w)
            # If it overflows vertically, instead fit by height
            if pic.height > avail_h:
                # Delete and re-add fitted by height
                slide.shapes._spTree.remove(pic._element)
                pic = slide.shapes.add_picture(stream, left, top, height=avail_h)

            # Center if there's leftover space
            # Horizontal centering
            pic.left = Inches(margin_in) + (avail_w - pic.width) // 2
            # Vertical centering
            pic.top  = Inches(margin_in) + (avail_h - pic.height) // 2

        elif mode == "text":
            title, body = _guess_title_and_body_from_page(page)

            slide_w = prs.slide_width
            slide_h = prs.slide_height

            # Title box
            if title:
                title_box = slide.shapes.add_textbox(
                    Inches(margin_in),
                    Inches(margin_in),
                    slide_w - Inches(margin_in*2),
                    Inches(1.0),
                )
                tf = title_box.text_frame
                tf.clear()
                p = tf.paragraphs[0]
                run = p.add_run()
                run.text = title
                run.font.size = Pt(36)  # big, editable
                p.space_after = Pt(6)

                # Body box beneath title
                body_top_in = margin_in + 1.0
            else:
                body_top_in = margin_in

            body_box = slide.shapes.add_textbox(
                Inches(margin_in),
                Inches(body_top_in),
                slide_w - Inches(margin_in*2),
                slide_h - Inches(body_top_in + margin_in),
            )
            tfb = body_box.text_frame
            tfb.word_wrap = True
            tfb.clear()
            if body:
                # Split into paragraphs on newlines
                parts = [b.strip() for b in body.split("\n") if b.strip()]
                if parts:
                    # First paragraph initializes the frame
                    p = tfb.paragraphs[0]
                    p.text = parts[0]
                    p.font.size = Pt(18)
                    for seg in parts[1:]:
                        np = tfb.add_paragraph()
                        np.text = seg
                        np.font.size = Pt(18)
                else:
                    tfb.paragraphs[0].text = ""
            else:
                tfb.paragraphs[0].text = ""

        else:
            raise ValueError("mode must be 'image' or 'text'")

    prs.save(pptx_path)
    print(f"✅ Saved: {pptx_path}")



In [ ]:
### UPLOAD PDF AND RUN CONVERSION

from google.colab import files

uploaded = files.upload()  # pick your PDF
pdf_path = list(uploaded.keys())[0]
print("PDF uploaded:", pdf_path)

# ---- Choose your options here ----
mode = "image"          # "image" or "text"
dpi = 220              # used only for image mode
match_size = False     # True = each slide matches its PDF page size
margin_in = 0.5        # slide margins (inches)
transparent = False    # keep PDF transparency (image mode only)
layout = "widescreen"  # "widescreen" or "standard" when match_size=False
out_name = "converted.pptx"
# ---------------------------------

convert_pdf_to_pptx(
    pdf_path,
    pptx_path=out_name,
    mode=mode,
    dpi=dpi,
    match_size=match_size,
    margin_in=margin_in,
    transparent=transparent,
    layout=layout
)


Saving Location_vs_Measurement_Coordinate_Systems.pdf to Location_vs_Measurement_Coordinate_Systems.pdf
PDF uploaded: Location_vs_Measurement_Coordinate_Systems.pdf
✅ Saved: converted.pptx


In [ ]:
### DOWNLOAD POWERPOINT FILE

files.download("converted.pptx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CLEAN UP. DELETE ALL FILES FROM SESSION FOLDER FOR NEXT SESSION

import os, glob

for pattern in ("*.pdf", "*.pptx"):
    for f in glob.glob(pattern):
        try:
            os.remove(f)
        except Exception as e:
            print("Could not remove", f, e)

print("Removed all .pdf and .pptx files in /content")


Removed all .pdf and .pptx files in /content
